# Build a data scientist agent with Claude Managed Agents

## Introduction

A data analyst tells you what happened; a data scientist tells you what will happen and how confident you should be. In this cookbook you'll build an agent that takes a raw CSV and returns a full modeling deliverable: exploratory analysis with statistical tests, a trained and cross-validated predictive model, and an honest write-up of what the model can and can't do.

You'll run it on [Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview), Anthropic's hosted runtime for stateful, tool-using agents, built on four core concepts:

- **Agent**: the model, system prompt, tools, MCP servers, and skills
- **Environment**: a configured container template (packages, network access)
- **Session**: a running agent instance within an environment, performing a specific task and generating outputs
- **Events**: messages exchanged between your application and the agent (user turns, tool results, status updates)

An agent plus an environment gives you a session. You attach your data to it as resources, then drive it by sending events and reading back the stream.

### What you'll learn

By the end of this cookbook you'll be able to:

- Configure an environment with a scientific Python stack
- Write a system prompt that enforces statistical rigor (cross-validation, baselines, calibrated claims)
- Retrieve multiple artifacts from one session: a report, a serialized model, and a metrics file

### Prerequisites

- Python 3.11+
- An Anthropic API key from the [Console](https://platform.claude.com/settings/keys), set as `ANTHROPIC_API_KEY`

Install dependencies:

In [ ]:
%%capture
%pip install -q "anthropic>=0.91.0" python-dotenv

In [ ]:
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv, set_key

load_dotenv()
client = Anthropic()
MODEL = "claude-sonnet-4-6"

## 1. Create an environment

The **environment** declares the scientific stack up front — `scikit-learn`, `scipy`, and `statsmodels` on top of the analysis basics — so every session can go straight from EDA to model training without a `pip install` detour.

Networking is `unrestricted` here so the agent can load plotly from its CDN; for production use a [host allowlist](https://platform.claude.com/docs/en/managed-agents/environments) instead.

In [ ]:
env = client.beta.environments.create(
    name="cookbook-data-scientist-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
        "packages": {
            "type": "packages",
            "pip": [
                "pandas",
                "numpy",
                "scikit-learn",
                "scipy",
                "statsmodels",
                "plotly",
            ],
        },
    },
)

## 2. Create the agent

The system prompt is where the data scientist differs from a data analyst. It demands a hypothesis-first workflow, a dummy baseline before any real model, cross-validation instead of a single train/test split, and uncertainty language in every claim. It also fixes the output contract: three artifacts with known names, so the retrieval step never has to guess.

[`agent_toolset_20260401`](https://platform.claude.com/docs/en/managed-agents/tools) provides eight tools: `bash`, `read`, `write`, `edit`, `glob`, `grep`, `web_fetch`, and `web_search`. The two web tools are disabled because this analysis is fully offline.

In [ ]:
DATA_SCIENTIST_SYSTEM_PROMPT = """\
You are a senior data scientist producing a rigorous modeling deliverable.

## Method
- Start with EDA: distributions, missingness, class balance, leakage checks.
- State 2-3 explicit hypotheses, then test them (scipy/statsmodels) and
  report effect sizes and p-values, not just "significant".
- Always fit a DummyClassifier/DummyRegressor baseline first; every model
  must be compared against it.
- Use stratified k-fold cross-validation (k=5) for all reported metrics;
  report mean +/- std, never a single split.
- Hold out a final test set untouched until the very end.

## Honesty
- Calibrate every claim: "the model improves recall from X to Y" beats
  "the model works great".
- Include a limitations section: sample size, leakage risks, drift.

## Execution
- Write .py scripts and run them with `python3 script.py`.
- Set random_state=42 everywhere for reproducibility.

## Output (all to /mnt/session/outputs/)
1. `report.html` — self-contained, inline CSS, plotly charts embedded via
   `fig.to_html(include_plotlyjs=False, full_html=False)` with plotly
   loaded once from the CDN in <head>; template='simple_white'.
2. `model.pkl` — the final fitted sklearn Pipeline (joblib.dump).
3. `metrics.json` — CV metrics, test metrics, and the baseline's metrics.
Confirm "Saved: report.html, model.pkl, metrics.json" when done.
"""

agent = client.beta.agents.create(
    name="cookbook-data-scientist",
    model=MODEL,
    system=DATA_SCIENTIST_SYSTEM_PROMPT,
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
            ],
        }
    ],
)

## 3. Generate and upload a dataset

To keep this notebook self-contained we synthesize a small customer-churn dataset with a few real signals buried in it (tenure and support tickets drive churn) plus noise columns the agent should learn to discount. Swap in any CSV with a target column; the rest of the flow is identical.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

tenure = rng.integers(1, 72, n)
tickets = rng.poisson(2, n)
monthly = rng.normal(60, 20, n).round(2)
contract = rng.choice(["month-to-month", "one-year", "two-year"], n, p=[0.5, 0.3, 0.2])

# churn truly depends on tenure, tickets, and contract type
logit = -0.04 * tenure + 0.45 * tickets + (contract == "month-to-month") * 1.2 - 1.0
churn = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)

df = pd.DataFrame(
    {
        "customer_id": [f"C{i:04d}" for i in range(n)],
        "tenure_months": tenure,
        "support_tickets": tickets,
        "monthly_charge": monthly,
        "contract": contract,
        "favorite_color": rng.choice(["red", "blue", "green"], n),  # noise
        "churned": churn,
    }
)

DATA_PATH = Path("churn_data.csv")
df.to_csv(DATA_PATH, index=False)

with DATA_PATH.open("rb") as f:
    dataset = client.beta.files.upload(file=(DATA_PATH.name, f, "text/csv"))

print(f"Uploaded {DATA_PATH.name} ({dataset.size_bytes} bytes) as {dataset.id}")

## 4. Create a session and send the task

A **session** binds the agent to the environment and any mounted files. `resources` mounts the uploaded file at the given absolute path inside the container. After creating the session, send a `user.message` event with the task; the agent starts working immediately.

In [ ]:
MOUNT_PATH = f"/mnt/session/uploads/{DATA_PATH.name}"
RESOURCES = [{"type": "file", "file_id": dataset.id, "mount_path": MOUNT_PATH}]

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=RESOURCES,
    title="Churn modeling",
)

TASK_PROMPT = f"""\
Build a churn prediction model from {MOUNT_PATH}.

Columns: customer_id, tenure_months, support_tickets, monthly_charge,
contract, favorite_color, churned (target, 0/1).

Run EDA with hypothesis tests, train a baseline and at least two candidate
models, pick a winner via cross-validation, evaluate on a held-out test
set, and produce report.html, model.pkl, and metrics.json per your system
instructions. Call out any features that look like pure noise.
"""

client.beta.sessions.events.send(
    session.id,
    events=[
        {"type": "user.message", "content": [{"type": "text", "text": TASK_PROMPT}]},
    ],
)
print(f"Session {session.id} running")

## 5. Stream the run

Open the session in the [Console](https://platform.claude.com/) under **Sessions** to watch every event, tool call, and token count live. The helper below tails the same event stream, printing `agent.message` text and `agent.tool_use` calls as they arrive, and returns on `session.status_idle`.

In [ ]:
def wait_for_idle(session_id: str) -> None:
    for ev in client.beta.sessions.events.stream(session_id):
        t = ev.type
        if t == "agent.message":
            for block in ev.content:
                if block.type == "text":
                    text = block.text
                    print(text[:300] + ("..." if len(text) > 300 else ""))
        elif t in ("agent.tool_use", "agent.mcp_tool_use"):
            print(f"  [{ev.name}]")
        elif t == "session.status_idle":
            return
        elif t == "session.status_terminated":
            raise RuntimeError(
                "Session terminated before going idle. "
                f"Trace: https://platform.claude.com/sessions/{session_id}"
            )


wait_for_idle(session.id)

## 6. Retrieve the deliverables

Anything the agent writes to `/mnt/session/outputs/` is persisted and surfaced via the Files API with `scope_id=<session_id>`; files written elsewhere in the container are not. This session should yield three artifacts: `report.html`, `model.pkl`, and `metrics.json`.

The [Files API](https://platform.claude.com/docs/en/api/beta/files/list) is a separate feature in beta, so to use `scope_id` here you also need to pass the Managed Agents beta header.

In [ ]:
outputs = client.beta.files.list(scope_id=session.id, betas=["managed-agents-2026-04-01"])
for f in outputs.data:
    print(f.filename, f.size_bytes)

# The list also includes the mounted inputs; download everything the agent
# wrote to /mnt/session/outputs/.
INPUT_NAMES = {r["mount_path"].rsplit("/", 1)[-1] for r in RESOURCES}
downloaded = []
for f in outputs.data:
    if f.filename in INPUT_NAMES:
        continue
    content = client.beta.files.download(f.id)
    out_path = Path("outputs") / f.filename
    out_path.parent.mkdir(exist_ok=True)
    out_path.write_bytes(content.read())
    downloaded.append(f.filename)
print("Downloaded:", downloaded)

if "report.html" not in downloaded:
    raise RuntimeError(f"report.html not found. Files: {[f.filename for f in outputs.data]}")

## 7. Clean up and next steps

You create the agent and environment once and reuse them across runs; you create a new session per task. Archive the session to release its container, and save the IDs to `.env` for reuse.

> **Warning:** make sure `.env` is listed in `.gitignore` before running the next cell — never commit it.

In [ ]:
client.beta.sessions.archive(session.id)

set_key(".env", "DATA_SCIENTIST_ENV_ID", env.id)
set_key(".env", "DATA_SCIENTIST_AGENT_ID", agent.id)
set_key(".env", "DATA_SCIENTIST_AGENT_VERSION", str(agent.version))
print("Saved DATA_SCIENTIST_ENV_ID, DATA_SCIENTIST_AGENT_ID, DATA_SCIENTIST_AGENT_VERSION to .env")

You've built a data scientist agent end to end: a scientific-stack environment, a system prompt that enforces baselines, cross-validation, and calibrated claims, and a session that returned a report, a serialized model, and a metrics file.

From here:

- Open `outputs/report.html` to read the analysis, and load `outputs/model.pkl` with `joblib.load` to score new customers.
- Check `outputs/metrics.json` to compare the winner against the baseline.
- Try a regression dataset — the system prompt already covers both task types.